# Section 3: Niche Analysis

## Purpose
Quantify neighborhood composition and derive biologically interpretable niche labels across KS samples.


In [ ]:
# Notebook config: autoreload local code while iterating on niche-analysis workflows
%load_ext autoreload
%autoreload 2

## Setup


In [ ]:
# System utilities
import os
import pickle
from pathlib import Path
from datetime import datetime
import warnings

# Data handling and numerical computation
import numpy as np
import pandas as pd


# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn3
import matplotlib.colors as mcolors
from upsetplot import UpSet, from_memberships
from plot_utils import proportion
from utils import plot_upset
from utils import scatter_df

# Single-cell analysis and related packages
import anndata as ad
import scanpy as sc
import squidpy as sq
from scipy import sparse

import anndata
import matplotlib.pyplot as plt
import time


# Machine learning and clustering
from sklearn.neighbors import KernelDensity, KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import roc_auc_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import silhouette_score

# Parallel processing
from joblib import Parallel, delayed
# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='numpy')
warnings.filterwarnings('ignore', category=FutureWarning, module='scanpy')
warnings.filterwarnings('ignore', category=UserWarning, module='scanpy')
warnings.filterwarnings('ignore', category=UserWarning, module='numpy')
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Function to print the current time with a message
def print_with_time(message):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}")

## Data Loading and Preprocessing


In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828', 
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

adata = sc.read_h5ad(f'../data/KS_adata_preprocessed.h5ad')
adata.obsm["spatial"] = adata.obs[["local_x", "local_y"]].copy().to_numpy()

KS_lytic_genes = ['KSHV.ORF50', 'KSHV.ORF57', 'KSHV.ORF59', 'KSHV.K9', 'KSHV.ORF65']
KS_latent_genes = ['KSHV.ORF71','KSHV.ORF72','KSHV.ORF73',]
KS_K2_gene = ['KSHV_K2']

In [ ]:
# Compute neighborhood composition features (NCTC) from spatial nearest neighbors
nctc_neighbors = 30
n_clusters = 10

In [ ]:
# Inspect baseline cell-type frequencies before neighborhood composition analysis
cell_type_counts = adata.obs.cell_type.value_counts().reset_index()
cell_type_counts['percentage'] = np.round(cell_type_counts['count']/len(adata)*100, 2)

In [ ]:
# Define color palettes for cell subtypes and downstream niche visualizations
color_mapping_subtypes = {'Lymphatic Endothelial Cells': '#9f7704',
                          'Macrophages': '#786d7b',
                          'Vascular Endothelial Cells': '#a4e000',
                          'Secretory-reticular Fibroblasts': '#324708',
                          'Pericytes': '#0086df',
                          'Fibroblasts': '#c7d0c0',
                          'Pro-inflammatory Fibroblasts': '#5b9494',
                          'Myofibroblasts': '#003b64',
                          'Cd8 Exhausted': '#af0078',
                          'Cycling Cells': '#3e198d',
                          'T-cells': '#ffb6c2',
                          'Differentiated Keratinocytes': '#683300',
                          'Secretory-papillary Fibroblasts': '#ff1c6d',
                          'Keratinocytes': '#ffa66d',
                          'Cd4': '#900013',
                          'Dendritic cells': '#495dff',
                          'Spinous to Granular Cells': '#fec507',
                          'Pilosebaceous Cells': '#bbbde2',
                          'B-cells': '#f12d00',
                          'Melanocytes': '#d56ee6',
                          'Mesenchymal Fibroblasts': '#3d8e27',
                          'Cd4 Rgcc': '#be0013'}

## Visualization and QC


In [ ]:
# Plot the final UMAP
with plt.rc_context({"figure.figsize": (25, 25), "figure.dpi": (200)}):
    sc.pl.umap(adata, color=['cell_type'], legend_loc='on data', size=3, legend_fontsize='xx-large', legend_fontoutline=5)

In [ ]:
# Define helper functions for neighborhood cell-type composition calculations
def calculate_neighborhood_cell_composition(df, n_neighbors=200, ctype_col='cell_type', x='CenterX_global_px_zero', y='CenterY_global_px_zero'):
    """
    This function calculates the composition of cell types in the neighborhoods 
    of each cell in a given DataFrame.
    
    Parameters:
    - df: DataFrame containing cell data, including x and y coordinates and cell types.
    - n_neighbors: Number of nearest neighbors to consider for each cell. Default is 200.
    
    Returns:
    - df: DataFrame with added columns representing the composition of cell types 
          in the neighborhoods of each cell.
    """

    # Extracting coordinates and cell types from the DataFrame
    coords = df[[x, y]].values
    cell_types = df[ctype_col].values
    
    # Obtaining unique cell types and sorting them
    unique_cell_types = sorted(df[ctype_col].unique())

    # Initializing a NearestNeighbors object and fitting it to the data
    neigh = NearestNeighbors(n_neighbors=n_neighbors)
    neigh.fit(coords)
    
    # Finding the indices of nearest neighbors for each point
    _, neighbors_indices = neigh.kneighbors(coords)

    # Mapping cell types to indices for faster processing
    cell_type_to_index = {cell_type: i for i, cell_type in enumerate(unique_cell_types)}
    cell_type_indices = np.vectorize(cell_type_to_index.get)(cell_types)

    # Initializing an array to hold the counts of cell types in neighborhoods
    cell_composition_counts = np.zeros((len(df), len(unique_cell_types)))

    # Printing progress information
    # print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Calculating Neighborhood Cell Composition")

    # Counting the occurrences of each cell type in the neighborhoods
    for i, neighbors in enumerate(neighbors_indices):
        # Updating progress every 30000 iterations
        # if i % 30000 == 0:
        #     print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Processing cell ID: {i}")
        neighbor_types = cell_type_indices[neighbors]
        for neighbor_type in neighbor_types:
            cell_composition_counts[i, neighbor_type] += 1

    # Creating a DataFrame from the counts
    cell_composition_df = pd.DataFrame(cell_composition_counts, columns=[f'n_{ct}' for ct in unique_cell_types], index=df.index)
    cell_composition_df.replace(pd.NA, 0, inplace=True)
    
    # Joining the new DataFrame with the original one
    df = pd.merge(df, cell_composition_df, left_index=True, right_index=True)

    # Printing completion message
    # print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Completed Neighborhood Cell Composition")

    
    return df
    
def process_uid(uid, adata, n_neighbors=30, cell_type_col='cell_type'):
    # Extract the data for the specific UID
    adata_uid = adata[adata.obs.path_block_core == uid]
    
    # Calculate the neighborhood cell composition
    df_with_neigh_comp = calculate_neighborhood_cell_composition(adata_uid.obs.copy(), x='local_x', y='local_y', ctype_col=cell_type_col, n_neighbors=n_neighbors)
    
    # Return the result along with the UID for merging back into the main object
    return uid, df_with_neigh_comp

In [ ]:
# 1. Create a new column named cell_types_updated which is a copy of the cell_type_AUC column
adata.obs['cell_types_updated'] = adata.obs['cell_type_initial'].astype(str)
mask_diff_ker = adata.obs['cell_type'] == "Differentiated Keratinocytes"
adata.obs.loc[mask_diff_ker, "cell_types_updated"] = "Differentiated Keratinocytes"


# 2. For every row in cell_types_updated, if the corresponding row in the KSHV_positive column is True, 
#    then append "KSHV+ " to the value in cell_types_updated if the value is not in the specified list.
excluded_types = ["B-cells", "Pilosebaceous Cells", "Pericytes", "Keratinocytes", "Melanocytes", "Unknown", "Spinous to Granular Cells", "Differentiated Keratinocytes"]
mask_kshv_positive = adata.obs['KSHV_positive'] == True

# Create a mask for excluding the specified types
mask_excluded_types = adata.obs['cell_types_updated'].isin(excluded_types)

# Combine masks: KSHV_positive and not in excluded types
mask_final = mask_kshv_positive & ~mask_excluded_types

# Apply the condition to the entire series at once using .loc
adata.obs.loc[mask_final, 'cell_types_updated'] = "KSHV+ " + adata.obs.loc[mask_final, 'cell_types_updated']


In [ ]:
# Inspect distribution of updated cell-type labels after harmonization
adata.obs.cell_types_updated.value_counts().reset_index()

In [ ]:
# Convert updated cell-type labels to categorical for consistent downstream handling
adata.obs['cell_types_updated'] = adata.obs['cell_types_updated'].astype('category')

In [ ]:
# Re-check category counts after categorical conversion
adata.obs['cell_types_updated'].value_counts()

In [ ]:
# List unique updated cell-type categories used for niche derivation
adata.obs['cell_types_updated'].unique().tolist()

In [ ]:
# Define ordered cell-type display sequence for composition plots and summaries
cell_types_order = [
    'KSHV+ Lymphatic Endothelial Cells',
    'KSHV+ Proliferating Lymphatic Endothelial Cells',
    'KSHV+ Vascular Endothelial Cells',
    'KSHV+ Fibroblasts',
    'KSHV+ Macrophages',
    'KSHV+ Dendritic cells',
    'KSHV+ T-cells',
    "Lymphatic Endothelial Cells",
    'Proliferating Lymphatic Endothelial Cells',
    "Vascular Endothelial Cells",
    "Fibroblasts",
    "Pericytes",
    "Keratinocytes",
    'Differentiated Keratinocytes',
    "Spinous to Granular Cells",
    "Melanocytes",
    "Pilosebaceous Cells",
    "Macrophages",
    "Dendritic cells",
    "T-cells",
    "B-cells"
]
adata.obs['cell_types_updated'] = adata.obs['cell_types_updated'].cat.reorder_categories(cell_types_order, ordered=True)

adata.obs['cell_types_updated']

In [ ]:
%%time

# List of unique UIDs
unique_ids = list(adata.obs.path_block_core.unique())
nctc_neighbors = 30
n_clusters = 10
cell_type_col = 'cell_types_updated'

# Use joblib to parallelize the processing
results = Parallel(n_jobs=-1)(delayed(process_uid)(uid, adata, nctc_neighbors, cell_type_col) for uid in unique_ids)

## Niche Derivation and Output Export


In [ ]:
# Define the path to save the pickle file
filename = f'results/NCTC/nctc_results_{nctc_neighbors}_with_KSHV_status_updated_ctypes_v5_nov15.pickle'

# Open a file in binary-write mode
with open(filename, 'wb') as file:
    # Serialize the 'results' object and write it into the file
    pickle.dump(results, file)

print(f"'results' has been saved as a pickle file at {filename}.")

In [ ]:
%%time

# Open a file in binary-write mode
with open(f'results/NCTC/nctc_results_{nctc_neighbors}_with_KSHV_status_updated_ctypes_v5_nov15.pickle', 'rb') as file:
    # Serialize the 'results' object and write it into the file
    results = pickle.load(file)


for ctype in adata.obs.cell_types_updated.unique():
    try:
        adata.obs.drop(columns=['n_'+ctype], inplace=True)
    except:
        pass

for ctype in adata.obs.cell_types_updated.unique():
    try:
        adata.obs.drop(columns=['NCTC_'+ctype], inplace=True)
    except:
        pass

# Prepare to store original data types and handle new columns
original_dtypes = {col: adata.obs[col].dtype for col in adata.obs.columns}

# Integrate the results back into the original adata object
for uid, df_result in results:
    # Correctly obtain indices
    indices = adata.obs[adata.obs.path_block_core == uid].index
    
    # Convert categorical columns in df_result to strings (if needed)
    for col in df_result.columns:
        if col in adata.obs.columns and isinstance(adata.obs[col].dtype, pd.CategoricalDtype):
            df_result[col] = df_result[col].astype(str).copy()
            adata.obs[col] = adata.obs[col].astype(str).copy()
            
        elif col not in adata.obs.columns:
            # Add new column to adata.obs and initialize with default values or empty strings
            adata.obs[col] = 0
            # adata.obs[col] = adata.obs[col].astype(str)

    # Update and add new columns from df_result to adata.obs
    adata.obs.loc[indices, df_result.columns] = df_result.loc[indices, df_result.columns]

# Ensure all original columns are converted back to their original types
for col, dtype in original_dtypes.items():
    try:
        adata.obs[col] = adata.obs[col].astype(dtype).copy()
    except Exception as e:
        print(f"Failed to convert column {col} back to its original type: {dtype}")
        print(f"Error: {e}")

# Handle new columns by determining their optimal data types
for col in df_result.columns:
    if col not in original_dtypes:
        # Detect and set appropriate dtype for new columns
        try:
            # Attempt to convert to the best inferred data type
            adata.obs[col] = pd.to_numeric(adata.obs[col], errors='ignore')
            if adata.obs[col].dtype == object:
                adata.obs[col] = adata.obs[col].astype('category').copy()
        except Exception as e:
            print(f"Failed to set type for new column {col}")
            print(f"Error: {e}")

# Optionally clear temporary variables if needed
# del results, indices


In [ ]:
# Replace missing observation values before clustering and downstream modeling
adata.obs.replace(pd.NA, 0, inplace=True)
adata.obs.replace(np.nan, 0, inplace=True)

adata.obs[["n_"+ctype for ctype in adata.obs.cell_types_updated.unique()]].head()

In [ ]:
# Initialize GPU-accelerated tooling and prepare features for niche clustering
import cupy as cp
import cudf
import anndata
from cuml.cluster import KMeans
import matplotlib.pyplot as plt
import time


# Function to print the current time with a message
def print_with_time(message):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}")

# Start the process
print_with_time("Starting the process")

adata.obs['cell_types_updated'] = adata.obs['cell_types_updated'].astype('category')


# Extract the neighborhood compositions from the columns
print_with_time("Extracting unique cell types and composition columns")
cell_types = adata.obs['cell_types_updated'].cat.categories  # Use .cat.categories to get the unique categories
composition_columns = ["n_" + celltype for celltype in cell_types]

print_with_time("Extracting compositions from adata.obs")
compositions = adata.obs[composition_columns]
compositions = compositions.replace(pd.NA, 0.0)

# Transfer the data to GPU
compositions_gpu = cp.asarray(compositions.values)

In [ ]:
# Perform KMeans clustering
print_with_time(f"Performing KMeans clustering with {n_clusters} clusters")
kmeans = KMeans(n_clusters=n_clusters, random_state=1111, n_init=5)
kmeans.fit(compositions_gpu, )

# Retrieve the cluster labels
cluster_labels = kmeans.labels_

# Store the cluster labels in the original anndata object
adata.obs['niche_clusters'] = cluster_labels.get()
adata.obs['niche_clusters'] = adata.obs['niche_clusters'].astype('category')

In [ ]:
# Review niche-cluster membership counts after KMeans assignment
adata.obs.niche_clusters.value_counts()

In [ ]:
# Define ordered cell-type list for niche composition visualization
cell_types_order = [
    'KSHV+ Lymphatic Endothelial Cells',
    'KSHV+ Proliferating Lymphatic Endothelial Cells',
    'KSHV+ Vascular Endothelial Cells',
    'KSHV+ Fibroblasts',
    'KSHV+ Macrophages',
    'KSHV+ Dendritic cells',
    'KSHV+ T-cells',
    "Lymphatic Endothelial Cells",
    'Proliferating Lymphatic Endothelial Cells',
    "Vascular Endothelial Cells",
    "Fibroblasts",
    "Pericytes",
    "Keratinocytes",
    "Differentiated Keratinocytes",
    "Spinous to Granular Cells",
    "Melanocytes",
    "Pilosebaceous Cells",
    "Macrophages",
    "Dendritic cells",
    "T-cells",
    "B-cells"
]


adata.obs['cell_types_updated'] = adata.obs['cell_types_updated'].cat.reorder_categories(cell_types_order, ordered=True)

adata.obs['cell_types_updated']

In [ ]:
# Inspect raw niche cluster labels prior to biological remapping
adata.obs.niche_clusters

In [ ]:
# Original clusters and their new mappings
original_order = [7,2,0,1,8,9,4,5,3,6]
new_order = list(range(len(original_order)))  # This will be [0, 1, 2, 3, 4, 5, 6, 7, 8]

# Create the mapping dictionary
cluster_mapping = {original: new for original, new in zip(original_order, new_order)}

# Apply the mapping to rename the clusters
adata.obs['niche_clusters'] = adata.obs['niche_clusters'].map(cluster_mapping)

# Set the 'niche_clusters' column as a categorical type with the new order
adata.obs['niche_clusters'] = pd.Categorical(adata.obs['niche_clusters'], categories=new_order, ordered=True)

In [ ]:
# Map numeric niche clusters to interpretable biological niche names
niche_dict = {7: 'T cell stroma',
1: 'Stroma',
9: 'Differentiated epidermis',
0: 'VEC stroma',
3: 'Tumor Boundary',
2: 'Macrophage stroma',
8: 'Basal epidermis',
4: 'Tumor',
5: 'Tumor Core',
6: 'Immune'}
adata.obs['niches'] = adata.obs['niche_clusters'].map(niche_dict)
adata.obs.niches

## Separate VEC Stroma by Tumor Proximity


In [ ]:
# Configure output directory and compute per-core tumor-distance summaries
output_dir = 'results/tumor_proximity/'

In [ ]:
# Define tumor-proximity bins used to stratify cells by distance from tumor core
def assign_tumor_proximity(distance):
    if distance < 90:
        return 'inside'
    return 'outside'
adata.obs['tumor_proximity'] = adata.obs['distance_to_tumor_whole'].apply(assign_tumor_proximity)

In [ ]:
# Define per-core processing routine for tumor-proximity label assignment
def process_path_block(path_block, adata):
    core_adata = adata[adata.obs['path_block_core'] == path_block]
    # tumor_cells = core_adata.obs[core_adata.obs['niches'].isin(["Tumor Core"])][['x_centroid', 'y_centroid']].to_numpy()
    tumor_cells = core_adata.obs[core_adata.obs['niches'].isin(["Tumor Core", "Tumor", "Tumor Boundary"])][['x_centroid', 'y_centroid']].to_numpy()
    all_cells = core_adata.obs[['x_centroid', 'y_centroid']].to_numpy()
    sample_id = core_adata.obs['sample_id'].unique().tolist()[0]

    # Build KDTree for the cluster 5 cells
    kdtree = KDTree(tumor_cells)

    # Query the KDTree for the distance to the nearest cell in cluster 5 for each cell
    distances, _ = kdtree.query(all_cells)

    distances = pd.DataFrame({
        'cell_id': core_adata.obs.cell_id,
        'distance_to_tumor': distances,
        'path_block_core': [path_block]*len(distances),
        'sample_id': [sample_id]*len(distances)
    })
    return distances

In [ ]:
# Execute tumor-proximity assignment across path-block cores and collect outputs
path_block_cores = adata.obs['path_block_core'].unique()

distance_by_core = []
for path_block in tqdm(path_block_cores):
    distance_by_core.append(process_path_block(path_block, adata))

distance_by_core = pd.concat(distance_by_core)
distance_by_core.to_csv(output_dir/'distance_to_tumor_core_v6_n30_c10.csv', index=False)

In [ ]:
# Define final niche-label refinement that combines niche identity with proximity bins
def assign_niche_cluster(niche, distance):
    if niche == 'VEC stroma' and distance < 90:
        return f'TA VEC stroma'
    elif niche == 'VEC stroma':
        return f'VEC stroma'
    return niche
adata.obs['niche_with_tumor_proximity'] = adata.obs.apply(lambda x: assign_niche_cluster(x['niches'], x['distance_to_tumor_whole']), axis=1)